# Healthcare Agent Learning Lab 1/10

## LLM versus deterministic tool

**Suggested time:** 15 minutes  
**Demonstration type:** Mock or live model + deterministic computation

### Learning objectives

- Distinguish language generation from deterministic computation.
- Identify tasks that should be delegated to tools rather than free-text reasoning.
- Explain why a correct-looking answer is not the same as a verified answer.

### How to use this notebook

1. **Observe:** run the supplied demonstration and read the visible trajectory.
2. **Exercise:** discuss workflow, evidence, responsibility, and safety—no programming required.
3. **Try it:** edit only the clearly marked values in the exercise cell.
4. **Technical extension:** optional implementation work for technical participants.
5. **Reflect:** record what should be changed before the pattern is used in a real workflow.

> **Mode and safety notice:** Set `LAB_MODE` explicitly to `mock` or `live`; the notebook will stop rather than silently choose a mode. Mock mode is scripted and demonstrates system structure—it is not evidence of model performance. Live mode may vary between runs and may incur API cost. All patients, actions, protocols, and outputs in this curriculum are fictional or synthetic and must not be used for patient care.


## Demonstration
**Goal:** see *why tools, not a cleverer prompt, carry much of the value.*

We ask a small dosing arithmetic question two ways: the model alone, and the model + a calculator tool.
The model alone may get it right or wrong — and it won't tell you which. The tool makes the answer deterministic and checkable. (Runs in MOCK mode with no API key; set a key/model to make it real.)


> ### In plain terms
> A language model is good at *reasoning* but shaky at *exact arithmetic* — and it won't warn you when it slips. Below, it works out a weight-based drug dose on its own (and gets it slightly wrong), then gets it exactly right once we hand it a calculator.
>
> **The point:** reliability comes from giving the model trustworthy **tools**, not from wording the prompt more cleverly.
>
> **What you'll see:** two answers to the same dose question — "Model alone" (approximate, and it can't tell you it's off) vs "Model + tool" (exact and checkable).


In [ ]:

import os, json, re

# =============================================================
# Model backend — works two ways:
#   1) MOCK (explicit LAB_MODE=mock): no API key needed. Returns scripted responses
#      so you can run the whole notebook and see the STRUCTURE.
#   2) REAL model (explicit LAB_MODE=live): pip install openai, then either
#        - Cloud:  export OPENAI_API_KEY=sk-...        (uses OpenAI)
#        - Local open-weight (vLLM / LM Studio / Ollama):
#              export OPENAI_BASE_URL=http://localhost:8000/v1
#              export OPENAI_API_KEY=dummy
#              export MODEL=meta-llama/Llama-3.1-8B-Instruct   # your served model
# Everything below is model-agnostic: swap the model, keep the code.
# =============================================================
LAB_MODE = os.environ.get("LAB_MODE", "").strip().lower()
if LAB_MODE not in {"mock", "live"}:
    raise RuntimeError("Set LAB_MODE explicitly to 'mock' or 'live' before running this notebook.")
USE_MOCK = LAB_MODE == "mock"
MODEL    = os.environ.get("MODEL", "gpt-4o-mini")
if not USE_MOCK and not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("LAB_MODE=live requires OPENAI_API_KEY in the Jupyter process environment.")

def chat(messages, temperature=0):
    """Return the assistant's text for a list of {role, content} messages."""
    if USE_MOCK:
        return _mock(messages)
    from openai import OpenAI
    client = OpenAI(base_url=os.environ.get("OPENAI_BASE_URL"))  # None -> api.openai.com
    r = client.chat.completions.create(model=MODEL, messages=messages, temperature=temperature, timeout=60)
    usage = getattr(r, "usage", None)
    prompt_details = getattr(usage, "prompt_tokens_details", None) if usage else None
    completion_details = getattr(usage, "completion_tokens_details", None) if usage else None
    reported_cost = getattr(usage, "cost", None) if usage else None
    if reported_cost is None and getattr(usage, "model_extra", None):
        reported_cost = usage.model_extra.get("cost")
    globals()["_LAB_LAST_RESPONSE_META"] = {
        "response_id": getattr(r, "id", None),
        "input_tokens": int(getattr(usage, "prompt_tokens", 0) or 0),
        "cached_input_tokens": int(getattr(prompt_details, "cached_tokens", 0) or 0),
        "output_tokens": int(getattr(usage, "completion_tokens", 0) or 0),
        "reasoning_tokens": int(getattr(completion_details, "reasoning_tokens", 0) or 0),
        "total_tokens": int(getattr(usage, "total_tokens", 0) or 0),
        "reported_cost_usd": float(reported_cost) if reported_cost is not None else None,
    }
    return r.choices[0].message.content

print("Backend:", "MOCK (explicit scripted mode — no model calls)" if USE_MOCK else f"REAL model = {MODEL}")

def _mock(messages):
    """Scripted: the 'model alone' skips the lb->kg conversion (a realistic
    silent error); with a real model it may or may not — and it won't tell you
    which. The point is that the TOOL makes the answer deterministic & checkable."""
    last = messages[-1]["content"].lower()
    if "just the number" in last and "calculator" not in last:
        return "176"   # wrong: forgot to convert pounds to kilograms
    return "I need a calculator for that."


In [ ]:
# LEARNING-LAB OBSERVABILITY WRAPPER
# Records calls, latency, response IDs, token usage, and cost when available.
# Estimated cost uses the explicit teaching-lab rate table below; provider
# billing remains authoritative.
import time as _lab_time

_lab_raw_chat = chat
LAB_CALL_LOG = []
_LAB_LAST_RESPONSE_META = {}
LAB_PRICING_USD_PER_MILLION = {
    "gpt-6-luna": {"input": 0.10, "cached_input": 0.01, "output": 0.50},
    "openai/gpt-6-luna": {"input": 0.10, "cached_input": 0.01, "output": 0.50},
}

def chat(messages, temperature=0):
    global _LAB_LAST_RESPONSE_META
    _LAB_LAST_RESPONSE_META = {}
    started = _lab_time.perf_counter()
    answer = _lab_raw_chat(messages, temperature=temperature)
    meta = dict(_LAB_LAST_RESPONSE_META)
    input_tokens = int(meta.get("input_tokens", 0) or 0)
    cached_tokens = int(meta.get("cached_input_tokens", 0) or 0)
    output_tokens = int(meta.get("output_tokens", 0) or 0)
    pricing = LAB_PRICING_USD_PER_MILLION.get(MODEL)
    estimated_cost = None
    if pricing:
        estimated_cost = (
            max(input_tokens - cached_tokens, 0) * pricing["input"]
            + cached_tokens * pricing["cached_input"]
            + output_tokens * pricing["output"]
        ) / 1_000_000
    LAB_CALL_LOG.append({
        "mode": "mock" if USE_MOCK else "live",
        "model": MODEL,
        "temperature": temperature,
        "latency_seconds": round(_lab_time.perf_counter() - started, 4),
        "response_id": meta.get("response_id"),
        "input_tokens": input_tokens,
        "cached_input_tokens": cached_tokens,
        "output_tokens": output_tokens,
        "reasoning_tokens": int(meta.get("reasoning_tokens", 0) or 0),
        "total_tokens": int(meta.get("total_tokens", 0) or 0),
        "reported_cost_usd": meta.get("reported_cost_usd"),
        "estimated_cost_usd": estimated_cost,
    })
    return answer


### 1) The model alone
A weight-based dose: enoxaparin **1 mg/kg** for a patient weighing **176 lb**. Correct answer requires converting lb → kg (176 / 2.2046 = 79.8 kg → **79.8 mg**).


In [ ]:
q = ("A patient weighs 176 lb. Enoxaparin dose is 1 mg/kg. "
     "What is the dose in mg? Answer with just the number.")
ans = chat([{"role":"user","content":q}])
print("Model alone ->", ans, "mg")
print("Correct      -> 79.8 mg  (176 / 2.2046)")


### 2) Deterministic tool-assisted computation
The demonstration code performs the arithmetic with a calculator function. It illustrates the value of a tool, but the model does not yet select or invoke the tool autonomously; that behavior begins in Notebook 2.


In [ ]:
def calculator(expr: str) -> float:
    """A safe arithmetic tool (no eval; parses a math expression and computes it)."""
    import ast, operator as op
    ops={ast.Add:op.add,ast.Sub:op.sub,ast.Mult:op.mul,ast.Div:op.truediv,ast.USub:op.neg,ast.Pow:op.pow}
    def ev(n):
        if isinstance(n,ast.Constant): return n.value      # (was ast.Num — deprecated)
        if isinstance(n,ast.BinOp):    return ops[type(n.op)](ev(n.left),ev(n.right))
        if isinstance(n,ast.UnaryOp):  return ops[type(n.op)](ev(n.operand))
        raise ValueError("unsupported expression")
    return ev(ast.parse(expr,mode="eval").body)

# The "agent" step: decide the computation, then let the tool do it exactly.
dose = calculator("176 / 2.2046 * 1")
print(f"Model + tool -> {dose:.1f} mg   (grounded, reproducible, checkable)")


### Takeaway
The tool didn't make the model *smarter* — it made the result **reliable**. In a real workflow, the reliability of tools, retrieval/grounding, and verification usually matters more than which reasoning-style prompt you picked. This is the foundation the next two notebooks build on.

*Try:* set `OPENAI_API_KEY` (or point `OPENAI_BASE_URL` at a local open-weight model) and re-run — watch whether the model-alone answer changes run to run.


## Exercise

No programming is required for this section.

**Activity:** Classify four workflow steps—summarizing a note, calculating a unit conversion, checking a contraindication list, and drafting patient-friendly language—as primarily model, deterministic tool, retrieval, or human-review work. Discuss where two mechanisms should be combined.

### Discussion prompts

- Which part of the result can be independently verified?
- What could still go wrong even when the calculator is correct?
- What information should appear in an audit trace?

Record the group’s proposed workflow boundary, the evidence needed, and the person or role accountable for the consequential decision.


In [ ]:
# CLINICAL TRY-IT: edit only the labels on the right, then run this cell.
# Suggested labels: "model", "deterministic tool", "retrieval", "human review", or "combined".
TASK_CLASSIFICATION = {
    "Summarize a discharge note": "model",
    "Convert pounds to kilograms": "deterministic tool",
    "Find the current medication list": "retrieval",
    "Approve a medication-order change": "human review",
}
for task, owner in TASK_CLASSIFICATION.items():
    print(f"{task:42} -> {owner}")


## Optional technical extension

Replace the fixed arithmetic expression with a validated unit-conversion function. Add tests for kilograms, pounds, missing units, negative values, and malformed input.

Technical participants should report both successful behavior and at least one intentionally induced failure. Preserve the trace needed to explain why the failure occurred.


In [ ]:
# OBSERVABILITY SUMMARY
print("Model interaction summary")
print("  mode:", "mock" if USE_MOCK else "live")
print("  model:", MODEL)
print("  calls:", len(LAB_CALL_LOG))
print("  total measured latency (s):", round(sum(row["latency_seconds"] for row in LAB_CALL_LOG), 4))
print("  input tokens:", sum(row["input_tokens"] for row in LAB_CALL_LOG))
print("  cached input tokens:", sum(row["cached_input_tokens"] for row in LAB_CALL_LOG))
print("  output tokens:", sum(row["output_tokens"] for row in LAB_CALL_LOG))
print("  reasoning tokens:", sum(row["reasoning_tokens"] for row in LAB_CALL_LOG))
print("  total tokens:", sum(row["total_tokens"] for row in LAB_CALL_LOG))
reported = [row["reported_cost_usd"] for row in LAB_CALL_LOG if row["reported_cost_usd"] is not None]
estimated = [row["estimated_cost_usd"] for row in LAB_CALL_LOG if row["estimated_cost_usd"] is not None]
print("  provider-reported cost (USD):", round(sum(reported), 6) if reported else "unavailable")
print("  estimated cost (USD):", round(sum(estimated), 6) if estimated else "unavailable")
print("  response IDs captured:", sum(bool(row["response_id"]) for row in LAB_CALL_LOG))


## Interpretation boundary

The calculator improves arithmetic reproducibility, but it does not validate the clinical premise, units, patient identity, or whether the requested dose is appropriate. This notebook is tool-assisted computation, not yet autonomous tool selection.

### Exit question

In one sentence: **What should the environment enforce, rather than merely asking the model to do?**

Continue to the next notebook only after the group can distinguish the observed system behavior from any claim about clinical correctness.
